# Example 1: Regulon Validation and Eigengene Computation

This notebook demonstrates two functions from `utils.py`:
- `validate_coexpression()` — checks if a group of genes actually vary together
- `get_eigengene()` — summarises a regulon's activity per patient as a single score

## Background

A **regulon** is a group of genes believed to be co-regulated by a shared transcription factor.
Before using a regulon in downstream analysis, we validate that its genes actually co-vary
across patients using PCA. If the first principal component (PC1) explains ≥ 30% of variance,
the genes are actually moving together and the regulon is kept.

The **eigengene** is the PC1 score for each patient — a single number summarising how
active that regulon is in each patient. It is essentially a weighted expression score of all the genes in the regulon. High eigengene = genes in the regulon are highly expressed in that patient.

In [2]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath(r"D:\School\IITD\General\GBM\gbm_model.ipynb")))
from utils import validate_coexpression, get_eigengene

## 1. Create synthetic expression data

We simulate a small expression matrix: 20 genes × 50 patients.
Values are z-scored expression levels (mean=0, std=1 per gene across patients).

In [4]:
np.random.seed(42)
n_genes    = 20
n_patients = 50

gene_names    = [f'GENE_{i+1}' for i in range(n_genes)]
patient_names = [f'PATIENT_{i+1}' for i in range(n_patients)]

# Random expression matrix (genes x patients)
expr = pd.DataFrame(
    np.random.randn(n_genes, n_patients),
    index=gene_names,
    columns=patient_names
)

print(f"Expression matrix shape: {expr.shape}")
print(expr.iloc[:4, :4])   # preview

Expression matrix shape: (20, 50)
        PATIENT_1  PATIENT_2  PATIENT_3  PATIENT_4
GENE_1   0.496714  -0.138264   0.647689   1.523030
GENE_2   0.324084  -0.385082  -0.676922   0.611676
GENE_3  -1.415371  -0.420645  -0.342715  -0.802277
GENE_4   0.250493   0.346448  -0.680025   0.232254


## 2. Define two regulons

- **Coherent regulon**: genes that are strongly correlated (simulated to move together)
- **Incoherent regulon**: random genes with no shared pattern

We expect `validate_coexpression` to return `True` for the coherent regulon
and `False` for the incoherent one.

In [5]:
# Coherent regulon: inject a shared signal across 6 genes
shared_signal = np.random.randn(n_patients)   # one signal shared by all genes
for gene in ['GENE_0', 'GENE_1', 'GENE_2', 'GENE_3', 'GENE_4', 'GENE_5']:
    noise = np.random.randn(n_patients) * 0.3  # small noise
    expr.loc[gene] = shared_signal + noise

coherent_regulon   = ['GENE_0', 'GENE_1', 'GENE_2', 'GENE_3', 'GENE_4', 'GENE_5']
incoherent_regulon = ['GENE_10', 'GENE_11', 'GENE_12', 'GENE_13', 'GENE_14']

print(f"Coherent regulon:   {coherent_regulon}")
print(f"Incoherent regulon: {incoherent_regulon}")

Coherent regulon:   ['GENE_0', 'GENE_1', 'GENE_2', 'GENE_3', 'GENE_4', 'GENE_5']
Incoherent regulon: ['GENE_10', 'GENE_11', 'GENE_12', 'GENE_13', 'GENE_14']


## 3. Validate coexpression

`validate_coexpression(cluster, expr, threshold=0.30)` fits a 1-component PCA
on the regulon's submatrix and checks if PC1 explains ≥ 30% of variance.

- Returns `True`  → regulon is coherent, keep it
- Returns `False` → regulon has no dominant axis of variation, discard it

In [6]:
result_coherent   = validate_coexpression(coherent_regulon,   expr, threshold=0.30)
result_incoherent = validate_coexpression(incoherent_regulon, expr, threshold=0.30)

print(f"Coherent regulon   → valid: {result_coherent}")
print(f"Incoherent regulon → valid: {result_incoherent}")

# Expected output:
# Coherent regulon   → valid: True
# Incoherent regulon → valid: False

Coherent regulon   → valid: True
Incoherent regulon → valid: False


### Edge cases handled by the function

In [7]:
# Too few genes — PCA needs at least 2
print(validate_coexpression(['GENE_0'], expr))                    # False

# Genes not in expression matrix — filtered out silently
print(validate_coexpression(['FAKE_GENE', 'GENE_0'], expr))       # False (only 1 valid gene)

# Custom threshold — stricter
print(validate_coexpression(coherent_regulon, expr, threshold=0.90))

False
False
True


## 4. Compute eigengene

`get_eigengene(regulon, expr)` computes PC1 scores for each patient
and sign-corrects so that high eigengene = high expression of regulon genes.

**Input:**  a validated regulon (list of gene names) + expression matrix

**Output:** a `pd.Series` of length n_patients — one score per patient

In [9]:
eigen = get_eigengene(coherent_regulon, expr)

print(f"Eigengene type:  {type(eigen)}")
print(f"Eigengene shape: {eigen.shape}")
print(f"\nFirst 5 patient scores:")
print(eigen.head())
print(f"\nMean:  {eigen.mean():.4f}")
print(f"Std:   {eigen.std():.4f}")
print(f"Range: [{eigen.min():.4f}, {eigen.max():.4f}]")

Eigengene type:  <class 'pandas.core.series.Series'>
Eigengene shape: (50,)

First 5 patient scores:
PATIENT_1    2.491679
PATIENT_2    1.045141
PATIENT_3   -0.706886
PATIENT_4   -2.537061
PATIENT_5    1.363065
dtype: float64

Mean:  0.0000
Std:   2.3625
Range: [-5.0740, 4.4823]


### Verify sign correction

The eigengene should be positively correlated with mean expression
of the regulon genes — high eigengene = genes are highly expressed.

In [10]:
mean_expr = expr.loc[coherent_regulon].T.mean(axis=1)
correlation = np.corrcoef(eigen, mean_expr)[0, 1]
print(f"Correlation between eigengene and mean expression: {correlation:.4f}")
print(f"Should be positive (sign correction working): {correlation > 0}")

Correlation between eigengene and mean expression: 1.0000
Should be positive (sign correction working): True


## 5. Run on all regulons in a pipeline

In the full pipeline, we run both functions across all decomposed subgroups.

In [11]:
# Simulate 10 candidate regulons of varying sizes
all_clusters = [
    gene_names[i:i+np.random.randint(3, 8)] 
    for i in range(0, n_genes, 2)
]

validated  = []
eigengenes = {}

for idx, cluster in enumerate(all_clusters):
    if validate_coexpression(cluster, expr):
        validated.append(cluster)
        eigen = get_eigengene(cluster, expr)
        if eigen is not None:
            eigengenes[idx] = eigen

print(f"Total clusters:          {len(all_clusters)}")
print(f"Validated regulons:      {len(validated)}")
print(f"Eigengenes computed:     {len(eigengenes)}")

Total clusters:          10
Validated regulons:      6
Eigengenes computed:     6
